# Dijet Data/nPDF comparisons

Theory paths come from `hist_analysis.config.theory` through the shared registry; the supplied EPPS21 artifact includes eta1p9 and cl68 in its filename.

Select `NPDF_MODELS` and `COMPARISON_MODELS` in the configuration cell.
Stored EPPS21 × CT18 and nCTEQ15HQ are pPb/pp (full eta) and (pPb/pp) F/B; no Data/nPDF
ratio or absolute theoretical spectrum is constructed. PYTHIA references are
pure generator-level pp shapes. Experimental total uncertainties apply only
to Data and Data/PYTHIA. PYTHIA F/B bands show the stored statistical errors.


## Numerical definitions and intentional ratio-direction update

For a selected pTave interval, full-eta sample shapes are normalized independently. `ETA_NORMALIZATION='integral'` preserves unit bin-content sums; `'bin_width'` gives unit-area density. pTave spectra use the configured `NORMALIZATION_RANGE`. Neither operation determines a cross section.

Forward/backward is computed from **unnormalized** disjoint eta hemispheres: $F/B=N(+|\eta|)/N(-|\eta|)$. Dedicated stored F/B systematic and theory products are used; full-shape uncertainty is never reused for F/B.

All MC comparisons now use $R_i=D_i/M_i$, including pTave and official gen/reco. The legacy separate Data/gen and Data/reco figures are retained; the configured comparison additionally draws Data/gen and reco/gen together. Existing full/F/B Data/MC ratios retain their definition. For independent Data and MC statistical errors, ROOT propagates $\sigma_R^2=\sigma_D^2/M^2+D^2\sigma_M^2/M^4$. No binomial division or new covariance assumption is introduced.

Undefined zero-denominator bins retain ROOT's existing numerical behavior in exported products, but display copies omit those bins rather than showing artificial measured zeros. The horizontal reference line is one.


## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.data_distributions import forward_backward_from_full
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_data_file, save_histograms,
)
from hist_analysis.python.histogram_ops import normalize_histogram
from hist_analysis.python.plot_labels import CMS_PRELIMINARY_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.histogram_production import resolve_catalog_file
from hist_analysis.python.root_style import apply_curve_style, configure_cms_header

CMS_HEADER_LEFT = CMS_PRELIMINARY_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = True  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetPalette(ROOT.kBird)

## Configuration

Select `NPDF_MODELS` (EPPS21 × CT18 and/or nCTEQ15HQ)
and `COMPARISON_MODELS` (CP5, CUETP8M1, both or empty).
`DRAW_DATA_SYSTEMATIC` controls only stored experimental totals.
The seven exact nPDF intervals use MB below 80, Jet60 for 80–90 and 90–120,
and Jet100 for 120–160 and 200–300 GeV. Wider intervals are raw projections,
not averages of normalized shapes. Shared source loading caches each TH2 once.


In [ ]:
from hist_analysis.config.histograms import trigger_dijet_bins
DIJET_BINNING = 'normal'  # normal or test; trigger-specific shared test assignment
DRAW_OFFICIAL_MC = False  # official gen/reco comparison pair
DRAW_CP5 = False
DRAW_CUETP8M1 = False
DRAW_MC_FIGURES = False
ETA_CUT_VALUE = 1.9  # per-jet |etaCM| <= value; supported producer cuts only
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'  # embedding or pythia
MC_FILE_STEM = 'jetId'
MC_FILE = resolve_combined_file(BASE_DIR, GENERATOR, MC_FILE_STEM, require_exists=False)
CP5_INPUT_SOURCE = 'local'  # auto, cernbox, or local
CERNBOX_CP5_GEN_FILE = resolve_catalog_file('official_mc__pythia_cp5__pp', require_exists=False)
LOCAL_CP5_GEN_FILE = resolve_catalog_file('standalone__cp5__pp', require_exists=False)
EXPLICIT_CP5_GEN_FILE = os.environ.get('CP5_GEN_FILE')

if CP5_INPUT_SOURCE not in {'auto', 'cernbox', 'local'}:
    raise ValueError("CP5_INPUT_SOURCE must be 'auto', 'cernbox', or 'local'")
if EXPLICIT_CP5_GEN_FILE:
    CP5_GEN_FILE = Path(EXPLICIT_CP5_GEN_FILE).expanduser()
elif CP5_INPUT_SOURCE == 'cernbox':
    CP5_GEN_FILE = resolve_catalog_file('official_mc_pythia_cp5_pp', require_exists=False)
elif CP5_INPUT_SOURCE == 'local':
    CP5_GEN_FILE = resolve_catalog_file('standalone_cp5_pp', require_exists=False)
else:
    cp5_candidates = (CERNBOX_CP5_GEN_FILE, LOCAL_CP5_GEN_FILE)
    CP5_GEN_FILE = next((path for path in cp5_candidates if path.exists()), cp5_candidates[0])
CP5_GEN_HISTOGRAM = 'hGenDijetPtEtaCM'
CUEP8M1_GEN_FILE = Path((os.environ.get('CUEP8M1_GEN_FILE') or resolve_catalog_file('standalone__cuep8m1__pp', require_exists=False))).expanduser()
CUEP8M1_GEN_HISTOGRAM = 'hGenDijetPtEtaCM'
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
DATA_DIRECTION = 'combined'  # combined, Pbgoing, or pgoing
DATA_SELECTION = 'jetId'     # jetId, trkMax, or noSel
DATA_FILES = {
    trigger: resolve_data_file(DATA_DIR, trigger, DATA_DIRECTION, DATA_SELECTION)
    for trigger in ('mb', 'jet60', 'jet80', 'jet100')
}
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUTS = DIJET_ETA_CUTS  # shared current production cut order
ETA_CUT_INDEX = eta_cut_index(ETA_CUT_VALUE)
ETA_CUT_INDICES = (ETA_CUT_INDEX,)
PTAVE_BINS = {
    'mb': [(50, 60), (60, 70), (70, 80)],
    'jet60': [(80, 90), (90, 100)],
    'jet80': [(100, 110), (110, 120)],
    'jet100': [
        (120, 130), (130, 140), (140, 150), (150, 160),
        (160, 180), (180, 200), (200, 250), (250, 300), (300, 500),
    ],
}
PTAVE_BINS = trigger_dijet_bins(PTAVE_BINS, DIJET_BINNING)
NORMALIZATION_RANGE = (110.0, 130.0)
PTAVE_DISPLAY_RANGE = (40.0, 500.0)
RATIO_Y_RANGES = {
    'ptave_data_over_mc': (0.75, 1.25),
    'eta_data_over_mc': (0.75, 1.15),
    'forward_backward': (0.75, 1.25),
    'fb_data_over_mc': (0.8, 1.2),
}
ETA_NORMALIZATION = 'integral'  # integral or bin_width
REBIN_PTAVE = 1
REBIN_ETA = 2
DRAW_DATA_SYSTEMATIC = True  # stored total experimental uncertainty, display only
DATA_SYSTEMATICS_DIR = Path(os.environ.get('DIJET_TOTAL_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/systematics_total')) / DATA_SELECTION / DIJET_BINNING
DATA_SYSTEMATICS_CONFIGURATION = None  # optional exact cfg_<hash> if several variants exist
DRAW_GRID = True
LOG_Y = True
SAVE_PDF = True
SAVE_PNG = False
SAVE_RATIO_HISTOGRAMS = True
RATIO_ROOT_BASENAME = 'dijet_data_vs_mc_ratios'
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_DATA_VS_NPDF_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_data_vs_npdf',
))

OFFICIAL_LABEL = {'embedding': 'Embedding', 'pythia': 'PYTHIA'}[GENERATOR]
# Keep legacy calculation labels/ROOT names; translate only displayed labels.
MC_RECO_LABEL = 'MC Reco (MC Emb.)'
EMBEDDING_GEN_LABEL = 'MC Gen (MC Emb.)'
CUEP8M1_GEN_LABEL = 'PYTHIA CUEP8M1'
CP5_GEN_LABEL = 'PYTHIA CP5'
DATA_LABEL = 'Data'
EPPS21_LABEL = 'EPPS21 x CT18'
# One style map drives ROOT objects and their MPLHEP export.
CURVE_STYLES = {
    EPPS21_LABEL: (ROOT.kGreen + 2, None),  # uncertainty band only
    MC_RECO_LABEL: (ROOT.kMagenta + 2, 20),
    EMBEDDING_GEN_LABEL: (ROOT.kMagenta + 2, 24),
    CUEP8M1_GEN_LABEL: (ROOT.kBlue + 1, 24),
    CP5_GEN_LABEL: (ROOT.kRed + 1, 20),
    DATA_LABEL: (ROOT.kBlack, 21),
}
COMPARISON_SUFFIX = f'_{GENERATOR}_selected'
# Each enabled family produces its own overlay and Data/MC figure.
COMPARISONS = {}
if DRAW_OFFICIAL_MC:
    COMPARISONS['official_mc'] = (EMBEDDING_GEN_LABEL, MC_RECO_LABEL)
if DRAW_CP5:
    COMPARISONS['pythia_cp5'] = (CP5_GEN_LABEL,)
if DRAW_CUETP8M1:
    COMPARISONS['pythia_cuetp8m1'] = (CUEP8M1_GEN_LABEL,)

# Central model registry: IDs and labels are independent of legacy ROOT names.
MODEL_REGISTRY = {
    'pythia_cp5': dict(id=1, label=CP5_GEN_LABEL),
    'pythia_cuetp8m1': dict(id=2, label=CUEP8M1_GEN_LABEL),
}
DEFAULT_REFERENCE_MODEL = 'pythia_cp5'
NPDF_MODELS = ('epps21_ct18', 'ncteq15hq')  # either model independently, or both; stored PDF errors already at 68% CL
# COMPARISON_MODELS = ('pythia_cp5', 'pythia_cuetp8m1')
COMPARISON_MODELS = ('pythia_cp5',)
if not set(COMPARISON_MODELS) <= {'pythia_cp5', 'pythia_cuetp8m1'}:
    raise ValueError('nPDF references must be CP5 and/or CUETP8M1')
FULL_MODELS = COMPARISON_MODELS
FB_MODELS = COMPARISON_MODELS
RATIO_COMBINATIONS = tuple(('data', model) for model in COMPARISON_MODELS)
from hist_analysis.config.histograms import NPDF_TRIGGER_BINS
PTAVE_BINS = {sample: list(NPDF_TRIGGER_BINS.get(sample, ())) for sample in DATA_FILES}
# Optional additional stored interval; requires its own matching experimental
# totals when DRAW_DATA_SYSTEMATIC=True. Never average the 50–60/60–70 totals.
EXTRA_MB_PTAVE_BINS = ()  # e.g. ((50, 70),); seven established intervals by default
if any(interval != (50, 70) for interval in EXTRA_MB_PTAVE_BINS):
    raise ValueError('Only the additional stored (50, 70) interval is available')
PTAVE_BINS['mb'] = list(dict.fromkeys((*PTAVE_BINS['mb'], *EXTRA_MB_PTAVE_BINS)))
if DIJET_BINNING != 'normal':
    raise ValueError('Stored nPDF comparisons require exact normal intervals')
# Exact y-axis limits for combined AND separate eta figures; None = automatic.
# Example: {'full': (0.0, 0.15), 'ratio': (0.7, 1.3), 'fb': (0.7, 1.3)}
# PLOT_Y_LIMITS = {'full': None, 'ratio': None, 'fb': None}
PLOT_Y_LIMITS = {'full': (0.0, 0.15), 'ratio': (0.75, 1.25), 'fb': (0.75, 1.25)}
# EPPS21 remains in its explicitly labeled observable-specific figures below.
# Never form Data/EPPS21: the stored theory is a pPb/pp ratio.
for numerator, denominator in RATIO_COMBINATIONS:
    if 'epps21_ct18' in (numerator, denominator):
        raise ValueError('EPPS21 is a stored pPb/pp observable, not a Data/MC operand')
    if numerator not in {'data', *MODEL_REGISTRY} or denominator not in MODEL_REGISTRY:
        raise ValueError(f'Unknown ratio operands: {numerator}, {denominator}')
for category, limits in PLOT_Y_LIMITS.items():
    if limits is not None and (len(limits) != 2 or not all(map(math.isfinite, limits)) or limits[0] >= limits[1]):
        raise ValueError(f'Invalid y limits for {category}: {limits}')
for model in (*FULL_MODELS, *FB_MODELS):
    if model not in MODEL_REGISTRY or model == 'epps21_ct18':
        raise ValueError(f'{model}: use the separate stored-theory comparison')

if NORMALIZATION_RANGE[0] >= NORMALIZATION_RANGE[1]:
    raise ValueError('NORMALIZATION_RANGE must satisfy low < high')
required_ratio_ranges = {
    'ptave_data_over_mc', 'eta_data_over_mc', 'forward_backward', 'fb_data_over_mc',
}
if set(RATIO_Y_RANGES) != required_ratio_ranges:
    raise KeyError(f'RATIO_Y_RANGES must contain exactly {required_ratio_ranges}')
for ratio_name, (low, high) in RATIO_Y_RANGES.items():
    if low >= high:
        raise ValueError(f'RATIO_Y_RANGES[{ratio_name!r}] must satisfy low < high')
if ETA_NORMALIZATION not in {'integral', 'bin_width'}:
    raise ValueError("ETA_NORMALIZATION must be 'integral' or 'bin_width'")
ETA_Y_TITLE = (
    '1/N dN/d#eta_{CM}^{dijet}' if ETA_NORMALIZATION == 'bin_width'
    else '1/N dN/d#eta_{CM}^{dijet}'
)
if any(index < 0 or index >= len(ETA_CUTS) for index in ETA_CUT_INDICES):
    raise IndexError(f'Invalid ETA_CUT_INDICES: {ETA_CUT_INDICES}')
if set(PTAVE_BINS) != set(DATA_FILES):
    raise KeyError('PTAVE_BINS and DATA_FILES must contain the same trigger samples')
if any(isinstance(value, bool) or not isinstance(value, int) or value < 1
       for value in (REBIN_PTAVE, REBIN_ETA)):
    raise ValueError('REBIN_PTAVE and REBIN_ETA must be positive integers')
for sample, filename in DATA_FILES.items():
    if PTAVE_BINS[sample] and not filename.exists():
        raise FileNotFoundError(f'Missing required Data ROOT file: {filename}')
# Optional MC sources are checked against their exact keys below; never substitute.

MC_FILE, CUEP8M1_GEN_FILE, CP5_GEN_FILE, DATA_FILES
OUTPUT_DIR = OUTPUT_DIR / DIJET_BINNING

from hist_analysis.python.data_mc_plotting import load_data_total, selection_lines, accepted_eta_range, check_distinct_generator_inputs, check_distinct_generator_predictions, load_comparison_source, data_over_mc, draw_comparison_figure

# Explicit display mapping avoids changing calculation labels and ROOT key names.
DISPLAY_LABELS = {f'{MC_RECO_LABEL} / {EMBEDDING_GEN_LABEL}': f'{OFFICIAL_LABEL} reco / gen', MC_RECO_LABEL: f'{OFFICIAL_LABEL} reco',
                  EMBEDDING_GEN_LABEL: f'{OFFICIAL_LABEL} gen',
                  CUEP8M1_GEN_LABEL: 'PYTHIA CUETP8M1', CP5_GEN_LABEL: 'PYTHIA CP5',
                  'CUEP8M1': 'PYTHIA CUETP8M1', 'CP5': 'PYTHIA CP5',
                  'Data / CUEP8M1': 'Data / PYTHIA CUETP8M1',
                  'Data / CP5': 'Data / PYTHIA CP5'}
def display_label(label):
    if label in DISPLAY_LABELS:
        return DISPLAY_LABELS[label]
    for original in (MC_RECO_LABEL, EMBEDDING_GEN_LABEL, CUEP8M1_GEN_LABEL):
        label = label.replace(original, DISPLAY_LABELS[original])
    return label

_data_total_cache = {}
def data_totals(sample, interval, observable):
    if not DRAW_DATA_SYSTEMATIC:
        return None
    key = (sample, interval, observable)
    if key not in _data_total_cache:
        _data_total_cache[key] = load_data_total(
            DATA_SYSTEMATICS_DIR, interval, observable=observable,
            direction=DATA_DIRECTION, selection=DATA_SELECTION, binning=DIJET_BINNING,
            eta_cut=ETA_CUT_VALUE, sample=sample, configuration=DATA_SYSTEMATICS_CONFIGURATION)
    return _data_total_cache[key]


# Output identity uses the actual configured selections. Each execution gets
# dated renderer directories and unique filenames; earlier files are preserved.
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection, nominal_jet_eta_cut
from hist_analysis.python.output_naming import verified_reference_eta_cuts
_output_requested_models = set(FULL_MODELS) | set(FB_MODELS) | {m for pair in RATIO_COMBINATIONS for m in pair}
if DRAW_CP5:
    _output_requested_models.add('pythia_cp5')
if DRAW_CUETP8M1:
    _output_requested_models.add('pythia_cuetp8m1')
_output_reference_cuts = verified_reference_eta_cuts([
    (label, path, key) for model, label, path, key in (
        ('pythia_cp5', 'CP5GenCM', CP5_GEN_FILE, CP5_GEN_HISTOGRAM),
        ('pythia_cuetp8m1', 'CUETP8M1GenCM', CUEP8M1_GEN_FILE, CUEP8M1_GEN_HISTOGRAM),
    ) if model in _output_requested_models
])
_output_eta_cuts = {'CM': [ETA_CUTS[i] for i in ETA_CUT_INDICES], **_output_reference_cuts}
OUTPUT_SELECTIONS = OutputSelection(DATA_SELECTION, _output_eta_cuts, PTAVE_BINS, 'ptAve')
OUTPUT_DIR = OutputDirectory(OUTPUT_DIR, OUTPUT_SELECTIONS, workflow_root=Path(os.environ.get('DIJET_DATA_VS_NPDF_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_data_vs_npdf')))



## Histogram loading, preparation and plotting

### Loading and normalization

The following cell defines the preparation workflow. Each detached input TH2 is cached once per preparation-cell execution. pTave and eta projections use the same selected ROOT objects; overlay and ratio renderers never reload them. pTave selections retain the existing inward-offset, half-open ROOT-bin convention. Normalization and F/B folding are performed once before creating figures.

### Eta domain and valid points

The producer cuts both jet etaCM values and fills their arithmetic mean. A populated bin may straddle the numerical jet cut. The visible domain therefore ends at the **physical outer edge of accepted bins**, derived independently for full eta and F/B. With the default 1.9 cut and current 0.2 bins, these edges are [-2,2] and [0,2]. Histogram values, binning and jet cuts are unchanged. Graph points outside this domain and missing theory points are omitted; theory rectangles never bridge gaps.

### Uncertainty sources and brackets

Statistical errors remain the original TH1 errors. `load_data_total` reads the authoritative total CSV, audit and provenance produced by `04_systematics_total.ipynb`, requiring exact observable, interval, trigger, beam, selection, binning and eta metadata. It never combines components. A point $y$ receives an absolute total systematic error $|y|f$ from the stored fractional total $f$; a ratio receives $|D/M|f$. MC statistical errors remain in the ratio's statistical error bar; they are not added to the Data systematic brackets.

Systematics use **two open brackets**, one at each $y\pm\delta_{syst}$ endpoint. Each has short vertical inward-facing ends, fixed display-point dimensions, and the same color as its Data or ratio marker. Statistical bars are separate and drawn in front. Disabling `DRAW_DATA_SYSTEMATIC` removes only these brackets. No applicable stored pTave-spectrum total exists, so none is invented.

### Overlay figures and separate ratio figures

`_draw_comparisons` sends the same prepared histograms to two single-panel render calls. Each enabled family gets an overlay containing Data and its MC reference, followed by a separate Data/MC figure with the denominator named in its legend. Official MC includes open gen and filled reco circles; optional CP5 and CUETP8M1 are independent pairs. There are no closure insets or summary canvases.

### Common ROOT/MPLHEP style and annotations

Both backends consume the same ROOT histograms, uncertainty graphs, styles and ranges. Every sample legend entry is marker-only, with no connecting line or statistical error bars, in all overlay and ratio figures; the theory entry remains a filled band. MPLHEP retains the systematic notebooks' CMS initialization, 8-inch figures, annotation/legend size 12, axis titles 20/18, and tick labels 16. Data use foreground black squares, CP5 red filled circles, CUETP8M1 blue open circles, and official MC magenta open/filled circles. EPPS21 is a green filled band; nCTEQ15HQ is magenta. Both use stored final total bands without markers or a central line. PDF uncertainties are already at 68% CL; no further conversion is applied.

The compact top-left block contains italic *anti*-kT R=0.4 (no comma), configured leading/subleading pT cuts, eta and angular cuts, and the current pTave interval. Numerical thresholds come from `AnalysisContract.h`. Beam orientation, jet-selection name and trigger remain in analysis configuration/filenames and are deliberately absent from the block. Shared axes-relative positions and headroom keep annotations, CMS labels, legends and points apart.


In [ ]:
from hist_analysis.python.dijet_comparison import build_comparison_workflow
globals().update(build_comparison_workflow(globals()))


In [ ]:
eta_results_by_sample = {sample: analyze_eta_sample(sample) for sample in PTAVE_BINS}
full_eta_intervals = [dict(sample=sample, ptave_range=interval,
    result=results[(ETA_CUT_INDEX, interval)])
    for sample, results in eta_results_by_sample.items() for interval in PTAVE_BINS[sample]]


## Stored theory and PYTHIA statistical bands

Stored asymmetric theoretical total bands are unchanged. Full eta shows
Data/PYTHIA ratios beside pPb/pp. F/B shows Data points, PYTHIA statistical
bands without markers, and the explicitly labeled stored (pPb/pp) F/B.
Experimental brackets scale only Data or Data/PYTHIA. No Data/nPDF division.


In [ ]:
from hist_analysis.python.npdf_comparison import (
    prediction_registry, select_predictions, draw_npdf_comparisons,
)
NPDF_REGISTRY = prediction_registry(PROJECT_ROOT)
NPDF_REGISTRY['epps21_ct18']['color'] = ROOT.kGreen + 2
NPDF_REGISTRY['ncteq15hq']['color'] = ROOT.kMagenta + 1
PREDICTIONS = select_predictions(NPDF_REGISTRY, NPDF_MODELS)
NPDF_Y_RANGES = {'full': (0.72, 1.3), 'fb': (0.72, 1.3)}
npdf_canvases = {}
for entry in full_eta_intervals:
    npdf_canvases[entry['ptave_range']] = draw_npdf_comparisons(entry, PREDICTIONS, globals())
    for canvas in npdf_canvases[entry['ptave_range']].values():
        display(canvas)

# Numerical products are saved independently of display bands and brackets.
if SAVE_RATIO_HISTOGRAMS:
    products = []
    for entry in full_eta_intervals:
        low, high = entry['ptave_range']
        for family in ('histograms', 'forward_backward', 'ratios', 'fb_ratios'):
            for label, histogram in entry['result'][family].items():
                tag = ''.join(c.lower() if c.isalnum() else '_' for c in label)
                clone = histogram.Clone(f'hNpdfComparison_{family}_{entry["sample"]}_ptave_{low}_{high}_{tag}')
                clone.SetDirectory(0); products.append(clone)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_histograms(str(OUTPUT_DIR / 'dijet_data_vs_npdf_ratios.root'), products)
